# 🛰️ CVUSA RGBD Test Pipeline — Kaggle + Google Drive

Bu notebook Google Drive'daki zip dosyalarını Kaggle'a indirip test çalıştırır.

**Gereksinimler:**
- Kaggle → Settings → **Internet: ON**
- Kaggle → Settings → **GPU T4: ON**
- Kaggle → Secrets → `WANDB_API_KEY` ekli
- Google Drive'da veri seti zip'leri ve model klasörü mevcut

## 1. GPU ve Paket Kontrolü

In [ ]:
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

!pip install -q wandb scipy pyyaml timm

## 2. Google Drive Bağlantısı (rclone)

rclone, Kaggle'dan Google Drive'a bağlanmanın en güvenilir yoludur.
**İlk kez kurulum için** aşağıdaki adımı takip edin.

In [ ]:
# rclone kur
!curl https://rclone.org/install.sh | sudo bash 2>/dev/null | tail -5
!rclone version | head -1

In [ ]:
# ── rclone token alma (SADECE İLK SEFERINDE) ─────────────────────
# 
# Adım 1: Kendi bilgisayarınızda terminalda şunu çalıştırın:
#   rclone authorize "drive"
# Bu komut tarayıcıda Google hesabınızla giriş yapmanızı ister.
# Başarılı olunca terminalde uzun bir JSON token çıkar.
#
# Adım 2: O token'ı Kaggle Secrets'a ekleyin:
#   Kaggle → Add-ons → Secrets → + Add New Secret
#   Name: RCLONE_TOKEN
#   Value: (terminaldeki JSON'u buraya yapıştırın)
#
# Adım 3: Bu hücreyi çalıştırın.

import os, json
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# rclone config dosyasını oluştur
rclone_token = secrets.get_secret('RCLONE_TOKEN')

rclone_config = f"""[gdrive]
type = drive
scope = drive.readonly
token = {rclone_token}
"""

os.makedirs(os.path.expanduser('~/.config/rclone'), exist_ok=True)
with open(os.path.expanduser('~/.config/rclone/rclone.conf'), 'w') as f:
    f.write(rclone_config)

# Bağlantıyı test et
result = os.popen('rclone lsd gdrive: --max-depth 1 2>&1').read()
print(result)
print('✅ Google Drive bağlantısı başarılı!' if 'ERROR' not in result else '❌ Bağlantı hatası!')

## 3. Drive'dan Dosyaları İndir

Drive'daki dosya/klasör yollarını kendi Drive yapınıza göre güncelleyin.

**Drive yolu nasıl bulunur?**  
`rclone lsd gdrive:` ile kök klasörlerinizi listeleyin, sonra `gdrive:KlasorAdi/` şeklinde kullanın.

In [ ]:
# Drive'daki kök klasörleri listele
!rclone lsd gdrive: --max-depth 1

In [ ]:
# ── Kendi Drive yollarınızı buraya girin ─────────────────────────

# Örnek: 'gdrive:MyDrive/cvpr2017_cvusa.zip'
DRIVE_RGB_ZIP    = 'gdrive:MyDrive/cvpr2017_cvusa.zip'           # RGB veri seti zip
DRIVE_DEPTH_ZIP  = 'gdrive:MyDrive/cvpr2017_cvusa_depth.zip'     # Depth zip
DRIVE_MODEL_DIR  = 'gdrive:MyDrive/model/lpn_square_test'        # Model klasörü (zip değil)

# Yerel hedef dizinler
LOCAL_DATA   = '/kaggle/working/data'
LOCAL_MODEL  = '/kaggle/working/model'
os.makedirs(LOCAL_DATA, exist_ok=True)
os.makedirs(LOCAL_MODEL, exist_ok=True)

print('Yollar ayarlandı. Sonraki hücreyle indirmeye başlayın.')

In [ ]:
# ── RGB veri seti zip'ini indir ve aç ────────────────────────────
print('📥 RGB zip indiriliyor...')
!rclone copy "{DRIVE_RGB_ZIP}" {LOCAL_DATA}/ --progress

rgb_zip = os.path.join(LOCAL_DATA, os.path.basename(DRIVE_RGB_ZIP))
print(f'\n📦 Zip açılıyor: {rgb_zip}')
!unzip -q {rgb_zip} -d {LOCAL_DATA}/
print('✅ RGB veri seti hazır')

# Açılan klasörü bul
!ls {LOCAL_DATA}/

In [ ]:
# ── Depth zip'ini indir ve aç ────────────────────────────────────
print('📥 Depth zip indiriliyor...')
!rclone copy "{DRIVE_DEPTH_ZIP}" {LOCAL_DATA}/ --progress

depth_zip = os.path.join(LOCAL_DATA, os.path.basename(DRIVE_DEPTH_ZIP))
print(f'\n📦 Zip açılıyor: {depth_zip}')
!unzip -q {depth_zip} -d {LOCAL_DATA}/
print('✅ Depth veri seti hazır')

!ls {LOCAL_DATA}/

In [ ]:
# ── Model ağırlıklarını indir ─────────────────────────────────────
print('📥 Model dosyaları indiriliyor...')
!rclone copy "{DRIVE_MODEL_DIR}" {LOCAL_MODEL}/lpn_square_test/ --progress

print('\nModel dosyaları:')
!ls {LOCAL_MODEL}/lpn_square_test/

## 4. Repo'yu Klonla

In [ ]:
REPO_URL = 'https://github.com/KULLANICI/REPO.git'  # ← kendi repo URL'niz
WORK_DIR = '/kaggle/working/cross-view-geo-localization'

if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
else:
    !cd {WORK_DIR} && git pull --rebase

os.chdir(WORK_DIR)
print('Aktif dizin:', os.getcwd())

# Model klasörünü proje dizinine bağla (symlink — kopyalamak yerine)
model_link = os.path.join(WORK_DIR, 'model', 'lpn_square_test')
os.makedirs(os.path.join(WORK_DIR, 'model'), exist_ok=True)
if not os.path.exists(model_link):
    os.symlink(f'{LOCAL_MODEL}/lpn_square_test', model_link)
print('Model linki:', model_link, '→ exists:', os.path.exists(model_link))

## 5. Test Dizinini Kontrol Et

In [ ]:
# Açılan zip'in içinde test dizini nerede?
# cvpr2017_cvusa.zip açıldığında genellikle cvpr2017_cvusa/test/ olur

import glob

# Test dizinini otomatik bul
candidates = glob.glob(f'{LOCAL_DATA}/**/test', recursive=True)
print('Bulunan test dizinleri:')
for c in candidates:
    print(f'  {c}')
    for item in sorted(os.listdir(c)):
        full = os.path.join(c, item)
        n = len(os.listdir(full)) if os.path.isdir(full) else os.path.getsize(full)
        print(f'    {item}/ ({n})')

In [ ]:
# ── Yukarıdaki çıktıya göre bu yolları ayarlayın ────────────────

TEST_DIR       = f'{LOCAL_DATA}/cvpr2017_cvusa/test'        # ← güncelleyin
DEPTH_DIR      = f'{LOCAL_DATA}/cvpr2017_cvusa_depth/test'  # ← güncelleyin
QUERY_FOLDER   = 'query_satellite'   # test/ altındaki sorgu klasörü adı
GALLERY_FOLDER = 'gallery_drone'     # test/ altındaki galeri klasörü adı

# Doğrulama
for label, path in [
    ('TEST_DIR', TEST_DIR),
    ('DEPTH_DIR', DEPTH_DIR),
    ('QUERY', os.path.join(TEST_DIR, QUERY_FOLDER)),
    ('GALLERY', os.path.join(TEST_DIR, GALLERY_FOLDER)),
]:
    exists = os.path.exists(path)
    n = len(os.listdir(path)) if exists else 0
    print(f'  {label}: {"✅" if exists else "❌"}  {path}  ({n} öğe)')

## 6. wandb Login

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

try:
    wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print('✅ wandb login başarılı')
except Exception as e:
    print(f'⚠️ Secret bulunamadı: {e}')
    print('Manuel login için: wandb.login(key="API_KEY")')

## 7. Test Çalıştır

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, 'test_cvusa.py',
    '--name',           'lpn_square_test',
    '--test_dir',       TEST_DIR,
    '--depth_dir',      DEPTH_DIR,
    '--query_folder',   QUERY_FOLDER,
    '--gallery_folder', GALLERY_FOLDER,
    '--use_rgbd',
    '--which_epoch',    'last',
    '--gpu_ids',        '0',
    '--batchsize',      '32',
    '--use_wandb',
    '--wandb_project',  'cross-view-geo-localization',
    '--wandb_run_name', 'kaggle_lpn_square_test',
]

print('Komut:', ' '.join(cmd))
print('─' * 60)

# Satır satır çıktı al (buffer sorunu olmaz)
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print(f'\n─ Return code: {process.returncode}')

## 8. Sonuçlar

In [ ]:
result_txt = './model/lpn_square_test/result.txt'
if os.path.exists(result_txt):
    print('=== Kaydedilen Metrikler ===')
    with open(result_txt) as f:
        print(f.read())
else:
    print('result.txt bulunamadı.')

if os.path.exists('pytorch_result.mat'):
    import scipy.io
    mat = scipy.io.loadmat('pytorch_result.mat')
    print(f'query_f  : {mat["query_f"].shape}')
    print(f'gallery_f: {mat["gallery_f"].shape}')